### 필요 라이브러리 설치
1. 실행 중인 jupyter notebook 종료, 관리자 권한으로 cmd 실행
- conda activate [가상환경이름]
- pip install -r requirements.txt

2. 루트 디렉토리에 .env 파일 생성
- OpenAI API KEY를 아래 형식대로 입력 후 저장
- OPENAI_API_KEY=your_openai_api_key

In [ ]:
from dotenv import load_dotenv

load_dotenv()

#### ChatOpenAI
- OpenAI에서 개발한 대화형 AI와 상호작용하기 위한 표준화된 인터페이스를 제공한다.
- 단순 텍스트 완성이 아닌 '대화(Chat)' 형식에 최적화된 모델을 호출할 때 사용한다.

`model_name`
- 사용할 AI 모델의 구체적인 명칭 작성

`temperature`
- 범위: 0.0 ~ 2.0 (일반적으로 0~1 사이를 주로 사용)
- 0.1 (낮은 값): 모델이 가장 확률이 높은 답변을 선택한다. 결과가 매우 일관적이고 논리적이며, 코딩이나 사실 관계 확인 등 정확도가 중요한 작업에 적합하다.
- 0.9 (높은 값): 모델이 다양한 단어를 선택하여 창의적이고 의외성 있는 답변으로 응답한다. 소설 쓰기, 아이디어 생각 등에 적합하다.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="gpt-4.1-nano",
    # model_name="gpt-4.1-mini",
    temperature=0.1
)

question = "연 수익률이 6%일 때 내 자산이 2배가 되는 데 걸리는 시간을 계산해줘!"

response = llm.invoke(question)
print(f"[답변에 대한 정보]: {response}")

In [ ]:
print(response.content)

### LogProb

- 토큰이란 문장을 구성하는 개별 단어나 문자 등의 요소를 의미하고, 확률은 모델이 그 토큰을 예측할 확률을 나타낸다.
- 특정 단어가 올 확률을 0에서 1 사이의 값($0 \le p \le 1$)으로 계산하지만, 문장이 길어질 수록 숫자가 작아진다.  
따라서 로그를 취해 확률 100%(1.0)는 $\log(1.0) = 0$이 되고, 확률이 낮아질수록 $-1, -5, -10$처럼 음수로 표현된다.  
즉, 값이 0에 가까울수록 AI가 그 단어를 선택할 확신이 높았다고 판단할 수 있다.
- 모델이 얼마나 자기 답변에 확신을 가지고 있는지, 혹은 어떤 부분에서 답변을 망설였는지 분석하기 위해 사용하는 기술적인 지표이다.

In [ ]:
from langchain_openai import ChatOpenAI

llm_with_logprob = ChatOpenAI(
    model_name="gpt-4.1-nano",
    # model_name="gpt-4.1-mini",
    temperature=0.1,
    max_tokens=500
).bind(logprobs=True, top_logprobs=5)

question = "대한민국의 수도는? 단답형으로 대답하라!"

response = llm_with_logprob.invoke(question)
print(f"[답변에 대한 확률]: {response.response_metadata}")

In [ ]:
import math

if "logprobs" in response.response_metadata:
    content_logprobs = response.response_metadata["logprobs"]["content"]
    
    for i, token_data in enumerate(content_logprobs):
        print(f"[{i+1}번째 토큰: '{token_data['token']}']에 대한 후보군:")
        print(f"{'-'*45}")
        print(f"{'후보 단어'}, {'확률(%)'}, {'LogProb'}")
        print(f"{'-'*45}")
        
        # 해당 토큰 위치에서 AI가 고려한 상위 5개 후보들
        for top_choice in token_data.get("top_logprobs", []):
            t = top_choice["token"]
            lp = top_choice["logprob"]
            prob = math.exp(lp) * 100
            
            print(f"{t}, {round(prob, 4)}%, {round(lp, 4)}")
        print("\n")

### 스트리밍 출력
| 구분 | `invoke` (일괄 처리) | `stream` (스트리밍) |
| :--- | :--- | :--- |
| **작동 방식** | 답변이 100% 완성될 때까지 기다린 후 한꺼번에 반환 | 첫 단어가 생성되는 즉시 한 조각씩 실시간 전송 |
| **체감 속도** | **느림** (전체 답변 생성 시간만큼 대기) | **매우 빠름** (첫 글자가 나오는 즉시 출력 시작) |
| **적합한 상황** | 결과값을 받아 후처리를 해야 하는 내부 로직 | 챗봇 UI, 긴 글 생성 등 사용자 대기 시간이 중요한 서비스 |
| **로컬 캐싱** | **쉬움** (질문-답변, 한 쌍을 그대로 매칭 가능) | **까다로움** (완료 전까지는 조각 단위라 캐싱 로직 필요) |

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano"
)

In [ ]:
# 스트림 방식으로 질의
answer = llm.stream("장난감을 주제로 짧은 시 한 수 지어줘")

In [ ]:
for chunk in answer:
    print(chunk.content, end="", flush=True)

In [ ]:
def stream_response(stream):
    for chunk in stream:
        print(chunk.content, end="", flush=True)

## 프롬프트 캐싱

프롬프트 캐싱 기능을 활용하면 반복하여 동일하게 입력으로 들어가는 토큰에 대한 비용을 아낄 수 있다.

`관련 문서: https://byline.network/2026/04/14-555/`

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-mini"
)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_content = """
당신의 임무는 주어진 질문에 친절하게 답변하는 것입니다.
아래에 주어진 정보를 참고하여 답변해 주세요.

정보: OpenAI가 내부 코딩 에이전트의 정렬 문제를 관찰하고 점검하는 방법
OpenAI는 모델의 역량이 발전함에 따라 행동과 내부 추론을 모두 포함한 에이전트 행동을 관찰하는 것이 점점 더 중요한 안전 도구가 될 것이라 보고 있습니다. 내부 코딩 에이전트를 관찰하면 정렬 문제가 발생하는 빈도와 문제가 실제 환경에서 나타나는 양상을 함께 이해할 수 있습니다. 실제 환경과 유사한 도구 중심 워크플로와 장시간 이어지는 세션에서 주로 나타나는 정렬 문제는 모델을 배포하기 전에 식별하기가 어렵습니다(개선을 위한 노력은 계속 진행 중입니다⁠(새 창에서 열기)). OpenAI의 목표는 이러한 문제와 행동을 정확하게 식별하는 것입니다. 또한 코딩 에이전트 세션에서 사용자 개인정보와 데이터를 보호하고, 문제 행동은 필요한 범위 내에서만 제한적으로 드러나도록 하는 것도 중요합니다.
OpenAI는 이를 위해 GPT‑5.4 Thinking(최대 추론 수준 적용)을 기반으로 하는 저지연 내부 모니터링 시스템을 구축했습니다. 이 시스템은 에이전트의 상호작용을 검토하고 사용자 의도와 어긋나거나 내부 보안 또는 컴플라이언스 정책을 위반할 수 있는 행동을 감지해 알려줍니다. 모니터링 시스템은 에이전트의 추론 과정과 행동을 기록 및 분석하고, 의심스럽거나 문제가 될 수 있는 행동을 자동으로 감지해 팀이 더 빠르게 대응하고 문제의 영향을 줄이며 다른 보호 장치를 개선할 수 있도록 지원합니다. 이러한 피드백 루프는 잠재적인 보안 문제를 신속히 탐지하고 모델의 정렬을 개선하는 데 기여하며, 장기적인 에이전트 보안 전략의 핵심 요소로 작용합니다.
현재 시스템은 상호작용이 완료된 후 30분 이내에 이를 검토하고, 유형을 분류한 뒤 심각도 수준을 지정합니다. 지연 시간이 줄어들어 실시간 검토에 가까워질수록 보안 효과는 크게 향상됩니다. 궁극적으로는 코딩 에이전트가 행동을 실행하기 전에 이를 평가해 기존 보안 모니터와 함께 중요한 다층 방어 수단으로 활용될 수 있습니다. 
초기 결과는 긍정적으로 나타났습니다. 모니터링 시스템은 직원이 내부 채널을 통해 별도로 보고한 모든 상호작용을 빠짐없이 감지하며, 직원 보고보다 일관되게 더 높은 성능을 보였습니다. 이와 함께 내부 사용자가 쉽게 알아차리지 못했을 다양한 추가 행동도 포착했습니다. 
OpenAI는 이를 통해 모델이 사용자 목표를 달성하려는 과정에서 제약을 우회하려 하는 경향이 있으며, 특히 사용자 요청이 이를 의도치 않게 유도할 때 이러한 경향이 강해진다는 점을 확인할 수 있었습니다. 다만 현재까지는 원래 작업을 넘어선 동기(예: 자기 보존이나 의도적 기만⁠)에 대한 증거는 확인되지 않았습니다. 전체 트래픽 중 극히 일부(<0.1%)는 맞춤형 또는 로컬 환경에서 발생한 것으로, 현재 모니터링 범위에 포함되지 않으며 팀은 이러한 공백을 해소하기 위해 노력하고 있습니다.
모니터링 시스템은 모든 표준 내부 코딩 에이전트의 배포 환경에서 비공개로 실행되며 모든 추론 과정과 사용자·개발자·어시스턴트의 메시지, 도구 호출 및 출력을 포함한 전체 대화 기록을 확인합니다. 모니터링 시스템은 사용자 의도와 어긋나거나 제약을 우회하려는 시도처럼 의심스러운 행동을 식별합니다. 잠재적인 이상 징후가 감지되면, 해당 내용은 사람이 직접 검토하고 대응할 수 있도록 담당자에게 전달됩니다. 이러한 방식은 사람의 검토 부담을 줄이면서도 모델의 정렬이나 행동에 영향을 줄 수 있는 문제를 적절하게 식별하고 대응할 수 있게 합니다.
현재 비동기 모니터링 인프라는 신뢰도를 계속 개선하는 단계에 있지만, 그럼에도 실제 내부 배포 환경에서는 모델의 행동을 관찰하고 이를 바탕으로 연구를 진행하는 데 실질적인 수단으로 활용되고 있습니다. OpenAI는 이를 프런티어 AI 시스템을 위한 보다 폭넓은 안전성 입증 체계⁠(새 창에서 열기)로 나아가는 초기 단계로 보고 있습니다. 이러한 체계는 모니터링, 평가, 통제를 결합해 모델 역량이 확장될수록 커지는 위험을 효과적으로 관리합니다.
성공적인 인프라를 구축하기 위한 로드맵은 다음 요소를 포함합니다.
적대적으로 구성된 데이터에서 모니터링 모델의 재현율을 추정하고, 모니터링 모델의 예측에 기반해 실제로 피해를 방지할 수 있는지 판단하는 통제 평가. 여기에는 모니터링 모델이 에이전트와 공모할 가능성을 평가하는 항목도 포함됩니다. 이를 위한 첫 단계로 진행된 실험에서는 OpenAI 코드베이스에 대한 PR을 은밀하게 훼손하려 하는 에이전트의 행동을 평가했습니다(자세한 내용은 Guan et al., 2025⁠(새 창에서 열기) 참고). 이 과제에서 최신 모니터링 시스템은 완벽한 성능을 달성했지만 여러 측면에서 볼 때 모니터링 모델에게는 지나치게 쉬운 과제였던 것으로 평가됩니다. OpenAI 팀은 현재 시스템의 한계를 계속해서 분석하고 있습니다.
더불어 모니터링 시스템과 인간 검토자에 대해 허용 가능한 신뢰도와 지연 시간 기준을 정립하고, 기반 모니터링 시스템의 개인정보 보호 및 보안 통제도 지속적으로 강화하고 있습니다.
모니터링을 동기식 차단 방식으로 확장하면 모델이 행동을 실제로 수행하기 전에 시스템이 이를 먼저 평가하고 차단할 수 있습니다.
"""
question = "OpenAI가 모니터링을 하는 이유가 무엇인가?"

messages = [
    SystemMessage(content=system_content),
    HumanMessage(content=question)
]

In [ ]:
from langchain.callbacks import get_openai_callback

with get_openai_callback() as cd:
    # 문자열이 아니라 메세지 리스트를 전달
    answer = llm.invoke(messages)

    # 캐싱 정보 확인
    usage = answer.response_metadata.get("token_usage", {})
    prompt_detail = usage.get("prompt_tokens_details", {})
    cached_tokens = prompt_detail.get("cached_tokens", 0)

    print(cd)
    print(f'캐싱된 토큰: {cached_tokens}')

In [ ]:
from langchain.callbacks import get_openai_callback

with get_openai_callback() as cd:
    # 문자열이 아니라 메세지 리스트를 전달
    answer = llm.invoke(messages)

    # 캐싱 정보 확인
    usage = answer.response_metadata.get("token_usage", {})
    prompt_detail = usage.get("prompt_tokens_details", {})
    cached_tokens = prompt_detail.get("cached_tokens", 0)

    print(cd)
    print(f'캐싱된 토큰: {cached_tokens}')

### InMemoryCache
- 로컬 RAM에 일시적으로 저장되며, dict자료형과 비슷하게 저장된다.
- key값이 정확히 동일해야 value를 가져올 수 있으므로 유사도 분석은 불가능하다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache
import time

# 로컬 메모리 캐시 활성화
set_llm_cache(InMemoryCache())

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-mini"    
)

start = time.time()
answer1 = llm.invoke("하늘이 왜 파란색이야?")
print(f"1차 응답 시간: {time.time() - start:.2f}s")
print(answer1.content)

print("-" * 30)

In [ ]:
start = time.time()
answer2 = llm.invoke("하늘이 왜 파란색이야?")
print(f"2차 응답 시간: {time.time() - start:.2f}s")
print(answer2.content)

print("-" * 30)

### Dict Stream Caching
- stream을 사용하면, 합쳐지기 전에 글자가 저장되기 때문에 다 합친 다음 직접 저장해야 한다.

In [ ]:
from langchain_openai import ChatOpenAI
import time

# 1. 객체 생성
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-mini"    
)

# 2. 로컬 메모리 캐시 저장소 (노트북 실행 동안 유지)
simple_cache = {}

def get_stream_with_cache(user_prompt):
    if user_prompt in simple_cache:
        print(f"\n캐시 히트\n======================")

        full_text = simple_cache[user_prompt]
        for char in full_text:
            yield char
            time.sleep(0.001) # 시각적 효과
        return


    print(f"\nAPI 호출\n======================")
    collected_content = ""

    for chunk in llm.stream(user_prompt):
        content = chunk.content
        if content:
            collected_content += content
            yield content

    # 스트리밍 완료 후 전체 답변 캐시에 저장
    simple_cache[user_prompt] = collected_content

In [ ]:
question = "개구리는 왜 녹색이야? 1문장으로 설명해라!"

print("--- 1차 실행 ---")
for chunk in get_stream_with_cache(question):
    print(chunk, end="", flush=True)
print("\n" + "-" * 30)

In [ ]:
question = "개구리는 왜 녹색이야? 1문장으로 설명해라!"

print("--- 2차 실행 ---")
for chunk in get_stream_with_cache(question):
    print(chunk, end="", flush=True)
print("\n" + "-" * 30)

### RedisCache

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisCache
from redis import Redis

# 1. Redis 연결 설정
# db=0: 기본값이 0, 0 ~ 15번까지 총 16개의 DB를 제공한다.
# DB 번호가 다르면 데이터가 서로 섞이지 않는다.
redis_client = Redis(host="localhost", port=6379, db=0)

# 2. 글로벌 캐시를 Redis로 설정
set_llm_cache(RedisCache(redis_client))

# 3. 모델 생성
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-mini"    
)

message = "폴란드의 수도는 어디야?"

In [ ]:
import time

def ask_question(message):
    start_time = time.time()
    response = llm.invoke(message)
    end_time = time.time()

    duration = end_time - start_time

    print(f'질문: {message}')
    print(f'답변: {response.content}')
    print(f'소요 시간: {duration:.4f}초')

    if duration < 0.1:
        print("캐시 히트")
    else:
        print("API 호출")
    print("-" * 30)    

In [ ]:
# 1회차: 서버에 다녀옴 (느림)
message = "폴란드의 수도는 어디야?"
ask_question(message)

In [ ]:
# 2회차: Redis에서 가져옴 (매우 빠름)
message = "폴란드의 수도는 어디야?"
ask_question(message)

## Redis Stream Caching
- Redis에서 Key를 먼저 조회하고, 없으면 API를 호출한다.
- API 호출 후 Redis에 캐싱해준다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisCache
from redis import Redis
import time

# 1. Redis 연결 설정
# decode_responses=True: 바이트 타입을 문자열(UTF-8)로 자동변환
redis_client = Redis(host="localhost", port=6379, db=0, decode_responses=True)

# 2. 모델 설정
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-mini"    
)

def ask_question_stream(message):
    start_time = time.time()
    # Redis 키 설정
    cache_key = f'llm_cache:{message}'

    # Redis에서 캐시 확인
    cached_response = redis_client.get(cache_key)

    if cached_response:
        print(f"질문: {message}")
        print(f"답변: ", end="", flush=True)
        for char in cached_response:
            print(char, end="", flush=True)
            time.sleep(0.001) # 시각적 효과

        duration = time.time() - start_time
        print(f"\n소요 시간: {duration:.4f}초")
        print("결과: 캐시 히트")
        print("-" * 30)
        return

    # API 호출 시: 실제 스트리밍 호출
    print(f"질문: {message}")
    print("답변: ", end="", flush=True)

    full_content = ""

    for chunk in llm.stream(message):
        content = chunk.content
        if content:
            full_content += content
            print(content, end="", flush=True)

    duration = time.time() - start_time

    if full_content:
        redis_client.setex(cache_key, 3600, full_content)

    print(f"\n소요 시간: {duration:.4f}초")
    print("결과: API 호출")
    print("-" * 30)

In [ ]:
# 1회차: API 호출 (느림, 이후 Redis에 저장됨)
ask_question_stream("폴란드의 수도는 어디야?")

In [ ]:
# 2회차: Redis에서 가져옴 (매우 빠름)
ask_question_stream("폴란드의 수도는 어디야?")

#### RedisSemanticCache
- 일반적인 RedisCache가 글자 하나하나가 똑같아야 하는 단어 일치 방식이라면, RedisSemanticCache는 질문의 의도를 파악하는 의미 일치 방식이다.
- 변환된 벡터 데이터(임베딩된 데이터)를 저장하고 검색하기 위해서는 Redis에 벡터 검색 엔진인 RediSearch 모듈이 탑재된 redis-stack이 필요하다.

1. 질문: 사용자가 "폴란드의 수도는 어디야?"라고 질문한다.
2. 임베딩: 허깅페이스 모델이 질문을 벡터 A로 변환한다.
3. 검색: Redis 내부에 저장된 기존 질문 벡터들과 벡터 A의 거리를 비교한다.
4. 판단:
- 가장 가까운 데이터와의 거리가 설정한 score_threshold(예: 0.2) 이내라면 캐시 히트
- 거리 안에 데이터가 없다면 API 호출 및 결과 저장
  
5. 반환: 캐시 히트 시 LLM 호출 없이 Redis에 저장된 답변을 즉시 반환한다.

`docker run -d --name redis-stack -p 6380:6379 -p 8001:8001 redis/redis-stack-server:latest`
- -d(detach): 실행만 시키고 뒤에서 돌리고 싶을 때
- -p 접속포트:내부실제포트 -p GUI접속포트:내부GUI실제포트

`docker exec -it redis-stack redis-cli`
- -i(interactive): 입력스트림을 열어서 키보드로 치는 명령어를 컨테이너가 받을 수 있게 함
-  -t(tty): 가상 터미널 할당(리눅스 터미널 환경처럼 보이게 한다)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import time
from langchain_openai import ChatOpenAI
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisSemanticCache
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.callbacks.manager import get_openai_callback

# 1. 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli",
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# 2. RedisSemanticCache 설정
REDIS_URL = "redis://localhost:6380"

set_llm_cache(
    RedisSemanticCache(
        redis_url=REDIS_URL,
        embedding=embeddings,
        score_threshold=0.35
    )
)

# 모델 설정
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-mini"
)

# 질문 함수 정의
def ask_question_redis_semantic(message):
    start_time = time.time()
    response = llm.invoke(message)
    end_time = time.time()

    duration = end_time - start_time

    print(f"질문: {message}")
    print(f"답변: {response.content}")
    print(f"소요 시간: {duration:.4f}초")

    if duration < 0.2:
        print("캐시 히트")
    else:
        print("API 호출")

In [ ]:
# 1회차: 서버에 다녀옴(느림, API 호출)
ask_question_redis_semantic("폴란드의 수도는 어디야?")

In [ ]:
# 2회차: 기존 문장과 똑같은 질문(매우 빠름, 캐시 히트)
ask_question_redis_semantic("폴란드의 수도는 어디야?")

In [ ]:
# 3회차: 문장이 조금 다른 유사 질문(매우 빠름, 캐시 히트)
ask_question_redis_semantic("폴란드의 수도는 어디라당가?")

###  RedisSemanticCache

In [ ]:
import time
from langchain_openai import ChatOpenAI
from langchain_community.cache import RedisSemanticCache
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Redis

# 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# RedisSemanticCache 설정
REDIS_URL = "redis://localhost:6380"
INDEX_NAME = "semantic_cache_index"

index_schema = {
    # text: 시맨틱 검색의 대상, 텍스트 분석이 필요하다고 요청
    # tag: 부가 내용, 문자열 그대로 유지, 이건 글자(텍스트)라고 알려줌
    "text": [{"name": "content"}] # 단어 형태나 의미로 쪼개서 검색할 대상(답변 내용)
    "tag": [{"name": "question"}] # 정확히 일치하는지 확인하는 카테고리나 식별자(질문 원문)
}

# 모델 설정
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-mini"
)

def ask_question_semantic_stream(message):
    try
        vector_db = Redis.from_existing_index(
            embeddings,
            index_name=INDEX_NAME,
            redis_url=REDIS_URL,
            schema=index_schema
        )
        results = vector_db.similarity_search_with_score(message)
    except Exception as e:
        print(e)
        results = []

    if results and results[0][1] <= score_threshold:
        doc, score = result[0]
        print(f"질문 : {message}")
        print(f"유사 질문 발견 : {doc.metadata['question']} 거리 : {score:.4f}")
        print("답변 :", end="", flush=True)

        cached_content = doc.page_content

        for char in cached_content:
            print(char, end="",flush=True)
            time.sleep(0.001)
            duration = time.time() - start_time
            print(f"\n소요 시간: {duration:.4f}초")
            print("결과: 캐시 히트")
            print("-" * 30)
